## Generating BJT using pure GDSfactory

In [ ]:
from glayout import MappedPDK, sky130 , gf180
#from gdsfactory.cell import cell
from gdsfactory import Component
from gdsfactory.geometry import boolean
from gdsfactory.components import text_freetype, rectangle, array, rectangular_ring

In [ ]:
top_level=Component(name="BJT")
pdk=gf180
pdk.activate()

In [ ]:
pdk.__dict__

In [ ]:
cdims=(8.28, 8.28)
bdims=(6.8,6.8)
edims=(5.32,5.32)
bjt_layers=("p+s/d","n+s/d","p+s/d")
bjt_dims=(edims,bdims,cdims)

In [ ]:
e_diff= rectangle(size=bjt_dims[0],layer=pdk.get_glayer(bjt_layers[0]),centered=True)
bt_diff= rectangle(size=bjt_dims[1],layer=pdk.get_glayer(bjt_layers[1]),centered=True)
ct_diff= rectangle(size=bjt_dims[2],layer=pdk.get_glayer(bjt_layers[2]),centered=True)
b_diff= boolean(A=bt_diff, B=e_diff, operation="not", 
                layer=pdk.get_glayer(bjt_layers[1]))
c_diff= boolean(A=ct_diff, B=bt_diff, operation="not",
                layer=pdk.get_glayer(bjt_layers[2]))
top_level.add_ref(e_diff)
top_level.add_ref(b_diff)
top_level.add_ref(c_diff)

In [ ]:
## Adding nwell
nw = rectangle(size=bjt_dims[1],layer=pdk.get_glayer("nwell"),centered=True)
top_level.add_ref(nw)

In [ ]:
comp_min_sep=0.16
contact_dims=(0.22, 0.22)
contact_inner_sep=0.25
contact_comp_sep=0.1

In [ ]:
import numpy as np

In [ ]:
## Adding comp
comp_layer_name="active_diff"
comp_e= rectangle(size=tuple(np.asarray(bjt_dims[0])-2*comp_min_sep),
                  layer=pdk.get_glayer(comp_layer_name),centered=True)
comp_out_b= rectangle(size=tuple(np.asarray(bjt_dims[1])-2*comp_min_sep),
                      layer=pdk.get_glayer(comp_layer_name),centered=True)
comp_inn_b= rectangle(size=tuple(np.asarray(bjt_dims[0])+2*comp_min_sep),
                      layer=pdk.get_glayer(comp_layer_name),centered=True)
comp_out_c= rectangle(size=tuple(np.asarray(bjt_dims[2])-2*comp_min_sep),
                      layer=pdk.get_glayer(comp_layer_name),centered=True)
comp_inn_c= rectangle(size=tuple(np.asarray(bjt_dims[1])+2*comp_min_sep),
                      layer=pdk.get_glayer(comp_layer_name),centered=True)

comp_b= boolean(A=comp_out_b, B=comp_inn_b, operation="not", 
                layer=pdk.get_glayer(comp_layer_name))
comp_c= boolean(A=comp_out_c, B=comp_inn_c, operation="not",
                layer=pdk.get_glayer(comp_layer_name))
top_level.add_ref(comp_e)
top_level.add_ref(comp_b)
top_level.add_ref(comp_c)

In [ ]:
#top_level.show()

In [ ]:
# Adding contacts
comp_min_sep=0.16
contact_dims=(0.22, 0.22)
contact_inner_sep=0.25
contact_array_sep=0.28
contact_comp_sep=0.1

contact_layer_name="mcon"
contact= rectangle(size=contact_dims,
                  layer=pdk.get_glayer(contact_layer_name),centered=True)

n_contacts_e=np.floor((np.asarray(edims)-
              2*comp_min_sep-
              2*contact_comp_sep-
              np.asarray(contact_dims)
             )/(np.asarray(contact_dims)+contact_array_sep)+
                      np.asarray((1,1)))

In [ ]:
n_contacts_b=np.floor((np.asarray(bdims)-
              2*comp_min_sep-
              2*contact_comp_sep-
              np.asarray(contact_dims)-
              np.asarray((1,0))*(np.asarray(bdims)-np.asarray(edims)-2*comp_min_sep)
             )/(np.asarray(contact_dims)+contact_inner_sep)+
                      np.asarray((1,1)))

n_contacts_c=np.floor((np.asarray(cdims)-
              2*comp_min_sep-
              2*contact_comp_sep-
              np.asarray(contact_dims)-
              np.asarray((1,0))*(np.asarray(cdims)-np.asarray(bdims)-2*comp_min_sep)
             )/(np.asarray(contact_dims)+contact_inner_sep)+
                      np.asarray((1,1)))

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
def round(x):
    return float(Decimal(x).quantize(Decimal("0.001"), 
                                 rounding=ROUND_HALF_UP)) 

def get_half_contact_array_size_m1(n_contacts,contact_inner_sep,contact_dims):
    return (n_contacts-1)*(contact_inner_sep +np.asarray(contact_dims))/2

def get_mid_ring(adims, bdims):
    return (np.asarray(adims)+np.asarray(bdims))/2 

def get_shift(position, n_contacts,contact_inner_sep, contact_dims, mid_ring):
    half_array_contact_size =  get_half_contact_array_size_m1(n_contacts,contact_inner_sep,contact_dims)
    
    if position=="top":
        hshift=(-1,0)*half_array_contact_size
        vshift=(0,1)*mid_ring/2
    elif position=="left":
        hshift=(-1,0)*mid_ring/2
        vshift=(0,-1)*half_array_contact_size
    elif position=="right":
        hshift=(1, 0)*mid_ring/2
        vshift=(0,-1)*half_array_contact_size
    elif position=="bottom":
        hshift=(-1, 0)*half_array_contact_size
        vshift=(0,-1)*mid_ring/2
    else:
        raise ValueError(f"Not a valid position: {position}")
    return hshift+vshift


In [ ]:
mid_ring_b_e=get_mid_ring(edims, bdims)
mid_ring_c_b=get_mid_ring(bdims, cdims)
get_shift("bottom", n_contacts_b,contact_inner_sep, contact_dims, mid_ring_b_e)

In [ ]:
def add_ring_contacts(parent, reference,n_contacts, mid_ring, contact_inner_sep,contact_dims):
    positions=["top", "bottom", "left", "right"]
    columns=[int(n_contacts[0]),int(n_contacts[0]),1,1]
    rows=[1, 1, int(n_contacts[1]),int(n_contacts[1])]
    
    refs=[]
    for position,n_column,n_row in list(zip(positions,columns,rows)):
    
        cref=parent.add_ref(reference, 
                         columns=n_column, 
                         rows=n_row, 
                         spacing=tuple(contact_inner_sep +np.asarray(contact_dims))
                        )
        cref.move(tuple([round(x)
               for x in get_shift(position, n_contacts,contact_inner_sep, contact_dims, mid_ring)]))
        refs.append(cref)
    return refs

def add_fill_contacts(parent, reference,n_contacts, contact_array_sep,contact_dims):

    ref=parent.add_ref(reference, 
                     columns=int(n_contacts[0]), 
                     rows=int(n_contacts[1]), 
                     spacing=tuple(contact_array_sep +np.asarray(contact_dims))
                    )
    ref.move(tuple(np.asarray((-1,-1))*(n_contacts-1)*(contact_array_sep +np.asarray(contact_dims))/2))
    return ref

In [ ]:
ccontacts=Component()
refs_e=add_fill_contacts(ccontacts,contact,n_contacts_e,contact_array_sep,contact_dims)
refs_b=add_ring_contacts(ccontacts,contact,n_contacts_b,mid_ring_b_e,contact_inner_sep,contact_dims)
refs_c=add_ring_contacts(ccontacts,contact,n_contacts_c,mid_ring_c_b,contact_inner_sep,contact_dims)


In [ ]:
ccontacts.show()

In [ ]:
top_level.add_ref(ccontacts)

In [ ]:
top_level.show()

In [ ]:
import gdsfactory as gf

In [ ]:
rd=gf.components.rectangular_ring(enclosed_size=edims,layer=pdk.get_glayer(contact_layer_name), width=0.74, centered=True)

In [ ]:
## Adding metal connection
metal_layer_name="met1"
metal_contacts=Component()
m1_e= rectangle(size=tuple(np.asarray(bjt_dims[0])-2*comp_min_sep),
                  layer=pdk.get_glayer(metal_layer_name),centered=True)

m1_b=rectangular_ring(enclosed_size=tuple(np.asarray(bjt_dims[0])+2*comp_min_sep),
                      layer=pdk.get_glayer(metal_layer_name), 
                      width=round((np.asarray(bjt_dims[1][0])-np.asarray(bjt_dims[0][0])-4*comp_min_sep)/2), 
                      centered=True)
m1_c=rectangular_ring(enclosed_size=tuple(np.asarray(bjt_dims[1])+2*comp_min_sep),
                      layer=pdk.get_glayer(metal_layer_name), 
                      width=round((np.asarray(bjt_dims[2][0])-np.asarray(bjt_dims[1][0])-4*comp_min_sep)/2), 
                      centered=True)
metal_contacts.add_ref(m1_e)
metal_contacts.add_ref(m1_b)
metal_contacts.add_ref(m1_c)

In [ ]:
metal_contacts.show()

In [ ]:
metal_contacts.add_label("E", position=(0,0),layer=pdk.get_glayer(metal_layer_name))

directions=((1,0),(0,1),(-1,0),(0,-1))
for direction in directions:
    metal_contacts.add_label("B", position=tuple(np.asarray(direction)*get_mid_ring(bdims,edims)/2),
                        layer=pdk.get_glayer(metal_layer_name))
    metal_contacts.add_label("C", position=tuple(np.asarray(direction)*get_mid_ring(cdims,bdims)/2),
                        layer=pdk.get_glayer(metal_layer_name))


In [ ]:
metal_contacts.show()

In [ ]:
top_level.add_ref(metal_contacts)

In [ ]:
top_level.show()

In [ ]:
drc_bjt= rectangle(size=tuple(np.asarray(bjt_dims[2])),
                  layer=(127,5),centered=True)
lvs_bjt= rectangle(size=tuple(np.asarray(bjt_dims[0])-2*comp_min_sep),
                  layer=(118,5),centered=True)

In [ ]:
drc_bjt.show()

In [ ]:
lvs_bjt.show()

In [ ]:
top_level.add_ref(drc_bjt)
top_level.add_ref(lvs_bjt)

In [ ]:
top_level.show()

In [ ]:
top_level.pprint_ports()

In [ ]:
bjt=top_level.flatten()

In [ ]:
bjt.show()

In [ ]:
bjt.pprint_ports()

In [ ]:
from pathlib import Path
import os
import subprocess
import tempfile
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"
magicrc_file

In [ ]:
def extract_pex(design, path_to_dir):
    design_name=design.name
    if not path_to_dir.exists():
        path_to_dir.mkdir(parents=True, exist_ok=False)
    
    pex_path = path_to_dir / f"{design_name}_pex.spice"
    gds_path = path_to_dir / f"{design_name}.gds"
    
    design.write_gds(str(gds_path))
        
    magic_script_content = f"""
    drc off            
    gds flatglob *\\$\\$*
    gds read {gds_path}
    
    flatten {design_name}
    load {design_name}
    select top cell
    extract do local
    extract all
    ext2sim labels on
    ext2sim
    extresist tolerance 10
    extresist
    ext2spice lvs
    ext2spice cthresh 0
    ext2spice extresist on
    ext2spice -o {str(pex_path)}
    exit
    """
    
    with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
        magic_script_file.write(magic_script_content)
        magic_script_path = magic_script_file.name
        
    magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
    magic_subproc = subprocess.run(
        magic_cmd, 
        shell=True,
        check=True,
        capture_output=True
    )
    
    magic_subproc_code = magic_subproc.returncode
    magic_subproc_out = magic_subproc.stdout.decode('utf-8')
    print(magic_subproc_out)

In [ ]:
path_to_dir = Path("/foss/designs/gLayout/tutorial").resolve() / "ext" / bjt.name
extract_pex(bjt, path_to_dir)